# VWAP BTCUSD Env + VM-Readiness Check (Beginner-Friendly)

This notebook is a **read-only safety check** before any VM reset or deployment.

It will:

1. Mount Google Drive
2. Clone or update the repo from GitHub
3. Check that `master-secrets.sops.yaml` and `age-keys.txt` exist in Drive
4. Install SOPS and age if needed
5. Render the `vwap_btcusd_dry_run` profile to an `.env` file in this Colab session
6. Verify only **variable names** and **non-secret safety flags** — never values
7. Run `scripts/secret_scan.py` and a `py_compile` check on the renderer
8. Print a final VM-reset readiness checklist

> **This notebook does not call Bybit, does not place orders, does not SSH, and does not reset the VM.**


---
## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print('Google Drive connected.')

---
## Step 2 — Clone or update the repo

Pulls the latest `main` branch into `/content/ict-trading-bot`.

In [ ]:
import os, subprocess, sys

REPO_DIR = '/content/ict-trading-bot'
REPO_URL = 'https://github.com/the-lizardking/ict-trading-bot.git'

def shell(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode != 0:
        if r.stderr.strip():
            print('Error:', r.stderr.strip())
        if check:
            sys.exit(r.returncode)
    return r

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo already cloned. Pulling latest ...')
    shell(f'git -C {REPO_DIR} pull --ff-only origin main')
else:
    print('Cloning repo ...')
    shell(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')

print(f'Repo ready at: {REPO_DIR}')

---
## Step 3 — Check Drive secrets are present

Verifies the encrypted master file and age key exist. **Does not read or print them.**

In [ ]:
import os, sys

SECRETS_DIR  = '/content/drive/MyDrive/ICT_Bot_Secrets'
MASTER_FILE  = os.path.join(SECRETS_DIR, 'master-secrets.sops.yaml')
AGE_KEY_FILE = os.path.join(SECRETS_DIR, 'age-keys.txt')

errors = []
if not os.path.isdir(SECRETS_DIR):
    errors.append(f'Folder not found: {SECRETS_DIR}')
else:
    print(f'[OK]  Folder found: {SECRETS_DIR}')

if not os.path.isfile(MASTER_FILE):
    errors.append(f'Encrypted file not found: {MASTER_FILE}')
else:
    print(f'[OK]  master-secrets.sops.yaml present ({os.path.getsize(MASTER_FILE)} bytes)')

if not os.path.isfile(AGE_KEY_FILE):
    errors.append(f'Age key file not found: {AGE_KEY_FILE}')
else:
    print('[OK]  age-keys.txt present')

if errors:
    print()
    for e in errors:
        print('ERROR:', e)
    sys.exit(1)

print()
print('All required files are present.')

---
## Step 4 — Install SOPS and age (if needed)

In [ ]:
import shutil, subprocess, sys

AGE_VERSION  = 'v1.1.1'
SOPS_VERSION = 'v3.8.1'

def shell(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode != 0:
        if r.stderr.strip():
            print('Error:', r.stderr.strip())
        if check:
            sys.exit(r.returncode)
    return r

if shutil.which('age'):
    print('age already installed:', shell('age --version', check=False).stdout.strip())
else:
    print(f'Installing age {AGE_VERSION} ...')
    shell(f'wget -q https://github.com/FiloSottile/age/releases/download/{AGE_VERSION}/age-{AGE_VERSION}-linux-amd64.tar.gz -O /tmp/age.tar.gz')
    shell('tar -xzf /tmp/age.tar.gz -C /tmp/')
    shell('cp /tmp/age/age /tmp/age/age-keygen /usr/local/bin/')
    shell('chmod +x /usr/local/bin/age /usr/local/bin/age-keygen')

if shutil.which('sops'):
    print('sops already installed:', shell('sops --version', check=False).stdout.strip())
else:
    print(f'Installing SOPS {SOPS_VERSION} ...')
    shell(f'wget -q https://github.com/getsops/sops/releases/download/{SOPS_VERSION}/sops-{SOPS_VERSION}.linux.amd64 -O /usr/local/bin/sops')
    shell('chmod +x /usr/local/bin/sops')

print('Tools ready.')

---
## Step 5 — Render the `vwap_btcusd_dry_run` profile

This generates `.env.vwap_btcusd_dry_run` inside the Colab repo workspace.
Secrets are never printed — only profile name, output path, and variable names.

In [ ]:
import os, subprocess, sys

SECRETS_DIR  = '/content/drive/MyDrive/ICT_Bot_Secrets'
MASTER_FILE  = os.path.join(SECRETS_DIR, 'master-secrets.sops.yaml')
AGE_KEY_FILE = os.path.join(SECRETS_DIR, 'age-keys.txt')
REPO_DIR     = '/content/ict-trading-bot'
SCRIPT       = os.path.join(REPO_DIR, 'scripts', 'render_env_from_master.py')
ENV_FILE     = os.path.join(REPO_DIR, '.env.vwap_btcusd_dry_run')

cmd = [
    sys.executable, SCRIPT,
    '--master', MASTER_FILE,
    '--age-key-file', AGE_KEY_FILE,
    '--profile', 'vwap_btcusd_dry_run',
    '--out', ENV_FILE,
]

result = subprocess.run(
    cmd, capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
)
if result.stdout.strip():
    print(result.stdout.strip())
if result.returncode != 0:
    if result.stderr.strip():
        print(result.stderr.strip())
    sys.exit(result.returncode)

print()
print(f'Env file generated: {ENV_FILE}')

---
## Step 6 — Verify variable names and safety flags only

Reads back the rendered `.env` and prints **only**:

- variable **names** (left of `=`)
- the values of known **non-secret safety flags** (`MODE`, `DRY_RUN`,
  `ALLOW_LIVE_TRADING`, `BYBIT_TESTNET`, `STRATEGY`, `SYMBOL`, `TIMEFRAME`,
  `EXCHANGE`, `ENVIRONMENT`)

It never prints API keys, secrets, or telegram tokens.

In [ ]:
SAFE_FLAGS = {
    'MODE', 'DRY_RUN', 'ALLOW_LIVE_TRADING', 'BYBIT_TESTNET',
    'STRATEGY', 'SYMBOL', 'TIMEFRAME', 'EXCHANGE', 'ENVIRONMENT',
}

names = []
flag_values = {}
with open(ENV_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' not in line:
            continue
        key, _, val = line.partition('=')
        names.append(key)
        if key in SAFE_FLAGS:
            v = val.strip().strip('"')
            flag_values[key] = v

print(f'Total variables : {len(names)}')
print('Variable names  :', ', '.join(names))
print()
print('Safety flag values (non-secret):')
for k in ['ENVIRONMENT', 'EXCHANGE', 'MODE', 'DRY_RUN', 'ALLOW_LIVE_TRADING',
          'BYBIT_TESTNET', 'STRATEGY', 'SYMBOL', 'TIMEFRAME']:
    if k in flag_values:
        print(f'  {k} = {flag_values[k]}')

assert flag_values.get('DRY_RUN') == 'true', 'DRY_RUN must be true for dry-run profile'
assert flag_values.get('ALLOW_LIVE_TRADING') == 'false', 'ALLOW_LIVE_TRADING must be false'
assert flag_values.get('BYBIT_TESTNET') == 'false', 'BYBIT_TESTNET expected to be false (live endpoint, dry-run only)'
assert flag_values.get('STRATEGY') == 'vwap', 'STRATEGY must be vwap'
assert flag_values.get('SYMBOL') == 'BTCUSD', 'SYMBOL must be BTCUSD'
print()
print('[OK] Safety flags verified.')

---
## Step 7 — Run the secret scanner and a syntax check

`secret_scan.py` checks for accidentally-committed secrets in the repo.
`py_compile` confirms the renderer script parses cleanly.

In [ ]:
import subprocess, sys

# Secret scan
r = subprocess.run([sys.executable, 'scripts/secret_scan.py'], cwd=REPO_DIR,
                   capture_output=True, text=True)
print('--- secret_scan.py ---')
if r.stdout.strip():
    print(r.stdout.strip())
if r.stderr.strip():
    print(r.stderr.strip())
print(f'(exit code: {r.returncode})')

# py_compile
r2 = subprocess.run([sys.executable, '-m', 'py_compile',
                     'scripts/render_env_from_master.py'],
                    cwd=REPO_DIR, capture_output=True, text=True)
print()
print('--- py_compile render_env_from_master.py ---')
if r2.stdout.strip():
    print(r2.stdout.strip())
if r2.stderr.strip():
    print(r2.stderr.strip())
print(f'(exit code: {r2.returncode})')

---
## Step 8 — Confirmation of what this notebook did NOT do

In [ ]:
print('This notebook did not call Bybit, did not place orders, did not SSH, and did not reset the VM.')

---
## Step 9 — VM-reset readiness checklist

Use this checklist before initiating any VM reset or redeploy.

- [x] VWAP env renders successfully (`.env.vwap_btcusd_dry_run` created above)
- [x] `DRY_RUN=true`
- [x] `ALLOW_LIVE_TRADING=false`
- [x] `BYBIT_TESTNET=false` acknowledged — live Bybit endpoint keys are used,
      but `DRY_RUN=true` prevents actual order placement
- [ ] **`STRATEGY=vwap` is not yet runtime-ready unless the VWAP strategy is
      actually implemented in `src/` and wired into the runtime loop. Confirm
      before deploying.**
- [ ] **VM audit still required before reset** — review running processes,
      pending deployments, in-flight orders, log retention, and any state on
      the VM before any reset/redeploy. This notebook does not perform that
      audit.

> Once these boxes are checked manually, proceed with the VM reset workflow
> documented in `docs/claude/deployment-ops.md` — separately from this notebook.
